Author: Daniel Abadjiev, with borrowed code from https://github.com/smart-pix/filter/blob/main/model_pipeline/model.py#L156 and from Eric You  
Date: Sep 25, 2025  
Description: Test out how hls4ml works

In [11]:
from OptimizedDataGenerator4 import *
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from qkeras import QDense, QActivation, QDenseBatchnorm
from qkeras.quantizers import quantized_bits, quantized_relu
import hls4ml

noGPU=False
if noGPU:
    tf.config.set_visible_devices([], 'GPU')

print(tf.config.experimental.list_physical_devices())
print(tf.test.is_built_with_cuda())
print(tf.test.is_built_with_gpu_support())
print(tf.test.is_gpu_available())

import os
import numpy as np
import tensorflow as tf
import csv
import pandas as pd
# import model as md
# import utils as ut

from qkeras import QDenseBatchnorm
import qkeras

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
True
True
False


2025-10-17 13:11:45.736738: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [14]:
filepath = "/home/dabadjiev/smartpixels_ml_dsabadjiev/erics_git/quantized_model1_results_20250922_162405/quantized_8w0i_8a0i/trial_1/quantized_mlp_quantized_8w0i_8a0i_trial1.keras"#"./DanielModels/model1.keras"
qmodel_file = "/local/d1/smartpixLab/fermiModels/ds8l6_padded_noscaling_qkeras_foldbatchnorm_d58w4a8model.h5"
# filepath = ""
co = {}       
qkeras.utils._add_supported_quantized_objects(co)
quantizedModel = tf.keras.models.load_model(qmodel_file,custom_objects=co,compile=True)
output_dir = "./hlsTmp"

In [ ]:
config = hls4ml.utils.config_from_keras_model(quantizedModel, granularity='name')
# Convert to an hls model

hls_model = hls4ml.converters.convert_from_keras_model(quantizedModel, hls_config=config, output_dir=output_dir,backend="Vitis")
hls_model.write()

Interpreting Model
Topology:
Layer name: input1, layer type: InputLayer, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: dense1, layer type: QDenseBatchnorm, input shapes: [[None, 16]], output shape: [None, 58]
Layer name: relu1, layer type: Activation, input shapes: [[None, 58]], output shape: [None, 58]
Layer name: dense2, layer type: QDense, input shapes: [[None, 58]], output shape: [None, 3]
Layer name: linear, layer type: Activation, input shapes: [[None, 3]], output shape: [None, 3]
Interpreting Model
Topology:
Layer name: input1, layer type: InputLayer, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: dense1, layer type: QDenseBatchnorm, input shapes: [[None, 16]], output shape: [None, 58]
Layer name: relu1, layer type: Activation, input shapes: [[None, 58]], output shape: [None, 58]
Layer name: dense2, layer type: QDense, input shapes: [[None, 58]], output shape: [None, 3]
Layer name: linear, layer type: Activation, input shapes: [[None, 3]], ou

In [20]:
hls_model.build(csim=False)


****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.1 (64-bit)
  **** SW Build 5069499 on May 21 2024
  **** IP Build 5075265 on Wed May 22 21:45:21 MDT 2024
  **** SharedData Build 5076995 on Wed May 22 18:29:18 MDT 2024
  **** Start of session at: Fri Oct 17 13:14:48 2025
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source /code/Xilinx_2024.1/Vitis_HLS/2024.1/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'dabadjiev' on host 'kdplab01' (Linux_x86_64 version 5.14.0-570.52.1.el9_6.x86_64) on Fri Oct 17 13:14:52 CDT 2025
INFO: [HLS 200-10] In directory '/home/dabadjiev/smartpixels_ml_dsabadjiev/smart-pixels-ml/hlsTmp'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Creating and opening project '/home/dabadjiev/smartpixels_ml_dsabadjiev/smart-pixels-ml/hlsTmp/myproject_prj'.
INFO: [HLS 200-1510] Ru

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '4.360',
  'BestLatency': '3',
  'WorstLatency': '3',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'FF': '1794',
  'LUT': '22540',
  'BRAM_18K': '0',
  'DSP': '0',
  'URAM': '0',
  'AvailableBRAM_18K': '5376',
  'AvailableDSP': '12288',
  'AvailableFF': '3456000',
  'AvailableLUT': '1728000',
  'AvailableURAM': '1280'}}